# Validación cruzada k-fold por vaca

Este notebook documenta la ejecución de la validación cruzada del dataset de BCS. Lee del bucket de S3 solo las tres tablas del dataset (no las fotos), muestra cómo quedan repartidas las vacas en los 5 pliegues y en cada rotación, comprueba que ninguna vaca quede en dos conjuntos a la vez y corre el modelo base en cada rotación.

El modelo base no mira las fotos. A todas las vacas de validación les asigna la mediana del BCS de las vacas de entrenamiento, así que sirve como piso para comparar los modelos de la fase de modelado y como prueba de que el esquema k-fold funciona de principio a fin.

Al final se descargan dos archivos con cifras agregadas, sin IDs de vacas, que van a la carpeta `reports/bigdata/resultados/` del repositorio. Los datos se quedan en Colab y no se suben a ningún lado.

## Configuración

Las llaves de AWS y el nombre del bucket se leen de los secretos de Colab `AWS_KEY_ID`, `AWS_SECRET` y `S3_BUCKET`. `PREFIJO_DATASET` es la carpeta del bucket donde está el dataset (`bcs/`). Si no es la correcta, la celda de descarga dice en qué carpetas del bucket sí hay un `indice_imagenes.csv`.

In [1]:
# Parámetros
PREFIJO_DATASET = "bcs/"
DESTINO = "/content/dataset"
K = 5
TOLERANCIAS = (0.25, 0.50)

In [2]:
!pip install -q boto3
import boto3
from botocore.exceptions import ClientError
from pathlib import Path
from google.colab import userdata

BUCKET = userdata.get("S3_BUCKET")
s3 = boto3.client("s3", region_name="mx-central-1",
                  aws_access_key_id=userdata.get("AWS_KEY_ID"),
                  aws_secret_access_key=userdata.get("AWS_SECRET"))


def listar(prefijo):
    # Llaves de los archivos del bucket bajo un prefijo
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for o in pagina.get("Contents", []):
            if not o["Key"].endswith("/"):
                yield o["Key"]


def bajar_dataset(destino):
    # Baja las tres tablas del dataset; si no están en PREFIJO_DATASET, dice dónde sí hay un índice
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    prefijo = PREFIJO_DATASET.strip("/") + "/"
    for nombre in ("indice_imagenes.csv", "vacas.csv", "evaluaciones.csv"):
        try:
            s3.download_file(BUCKET, prefijo + nombre, str(destino / nombre))
        except ClientError:
            otros = [k for k in listar("") if k.endswith("indice_imagenes.csv")]
            raise FileNotFoundError(f"No encontré {prefijo + nombre} en el bucket. Índices que sí existen: {otros}. "
                                    "Corrige PREFIJO_DATASET y vuelve a correr esta celda.")
    return destino


destino = bajar_dataset(DESTINO)
print("Tablas descargadas en", destino)

Tablas descargadas en /content/dataset


## Pliegues

Los pliegues vienen de la columna `fold` del índice y se asignaron en la preparación de los datos. La separación es por vaca, así que todas las fotos de una vaca están en el mismo pliegue. Las vacas se ordenaron por BCS (y por ID cuando empatan) y se repartieron en rotación, por eso cada pliegue tiene BCS bajos, medios y altos. Solo cuentan las fotos con `usar_en_entrenamiento = si`.

In [3]:
import numpy as np
import pandas as pd


def leer(nombre):
    return pd.read_csv(destino / nombre, dtype={"id_vaca": str, "fila_excel": str})


idx, vac, ev = leer("indice_imagenes.csv"), leer("vacas.csv"), leer("evaluaciones.csv")
usar = idx[idx["usar_en_entrenamiento"] == "si"].copy()
vac_e = vac[vac["en_entrenamiento"] == "si"].copy()
ev_e = ev[ev["en_entrenamiento"] == "si"].copy()
for df in (usar, vac_e, ev_e):
    df["fold"] = df["fold"].astype(int)

print(f"Fotos para entrenar: {len(usar)} de {usar['id_vaca'].nunique()} vacas | "
      f"vacas en entrenamiento: {len(vac_e)} | evaluaciones de esas vacas: {len(ev_e)}")

vista = lambda v: (lambda s: int((s == v).sum()))
pliegues = (usar.groupby("fold").agg(vacas=("id_vaca", "nunique"), fotos=("ruta", "size"),
                                     posterior=("vista", vista("posterior")),
                                     lateral_izq=("vista", vista("lateral_izq")),
                                     lateral_der=("vista", vista("lateral_der")))
            .join(vac_e.groupby("fold")["bcs_entrenamiento"].agg(bcs_medio="mean", bcs_min="min", bcs_max="max"))
            .join(ev_e.groupby("fold").size().rename("evaluaciones")))
pliegues.index.name = "pliegue"
display(pliegues.round(2))

Fotos para entrenar: 551 de 76 vacas | vacas en entrenamiento: 76 | evaluaciones de esas vacas: 106


,vacas,fotos,posterior,lateral_izq,lateral_der,bcs_medio,bcs_min,bcs_max,evaluaciones
pliegue,,,,,,,,,
0,16,146,67,43,36,2.99,2.00,3.75,25
1,15,69,42,16,11,2.96,2.25,3.75,21
2,15,113,49,25,39,2.98,2.25,3.75,19
3,15,138,61,39,38,3.02,2.50,3.75,22
4,15,85,38,23,24,3.02,2.50,3.75,19


## Rotaciones

Usamos el mismo esquema de la fase de modelado. En la rotación k, el pliegue k no se usa, el pliegue (k + 1) mod 5 es la validación (solo mide) y los otros tres son el entrenamiento. Al juntar las 5 rotaciones, cada vaca queda exactamente una vez en validación, y de ahí sale la cifra sobre el total de vacas.

In [4]:
def conjuntos(k):
    val = (k + 1) % K
    return k, val, [j for j in range(K) if j not in (k, val)]


vacas_pliegue = {k: set(vac_e.loc[vac_e["fold"] == k, "id_vaca"]) for k in range(K)}
filas = []
for k in range(K):
    sin_usar, val, ent = conjuntos(k)
    filas.append({"rotacion": k, "pliegue sin usar": sin_usar, "pliegue de validación": val,
                  "pliegues de entrenamiento": ", ".join(map(str, ent)),
                  "vacas de entrenamiento": sum(len(vacas_pliegue[j]) for j in ent),
                  "fotos de entrenamiento": int(usar["fold"].isin(ent).sum()),
                  "vacas de validación": len(vacas_pliegue[val]),
                  "fotos de validación": int((usar["fold"] == val).sum())})
rotaciones = pd.DataFrame(filas).set_index("rotacion")
display(rotaciones)

,pliegue sin usar,pliegue de validación,pliegues de entrenamiento,vacas de entrenamiento,fotos de entrenamiento,vacas de validación,fotos de validación
rotacion,,,,,,,
0,0,1,"2, 3, 4",45,336,15,69
1,1,2,"0, 3, 4",46,369,15,113
2,2,3,"0, 1, 4",46,300,15,138
3,3,4,"0, 1, 2",46,328,15,85
4,4,0,"1, 2, 3",45,320,16,146


## Revisión de fuga de datos

Si una vaca tuviera fotos en entrenamiento y en validación, el modelo podría aprender a reconocerla en lugar de aprender su BCS y las métricas saldrían infladas. Por eso revisamos lo siguiente.

1. Que cada vaca tenga un solo pliegue en el índice y que sea el mismo de `vacas.csv`.
2. Que cada evaluación de `evaluaciones.csv` tenga el pliegue de su vaca.
3. Que ninguna foto idéntica (misma huella SHA-256) quede en dos pliegues.
4. Que en cada rotación los tres conjuntos no compartan vacas y que entre los tres sumen todas las vacas.
5. Que cada vaca quede en validación exactamente una vez en las 5 rotaciones.

Si algo falla, la celda se detiene y dice qué fue.

In [5]:
problemas = []
fold_vaca = vac_e.set_index("id_vaca")["fold"]

varios = usar.groupby("id_vaca")["fold"].nunique()
if (varios > 1).any():
    problemas.append(f"{int((varios > 1).sum())} vacas con fotos en más de un pliegue")
distinto = int(usar["fold"].ne(usar["id_vaca"].map(fold_vaca)).sum())
if distinto:
    problemas.append(f"{distinto} fotos con un pliegue distinto al de su vaca en vacas.csv")
distinto_ev = int(ev_e["fold"].ne(ev_e["id_vaca"].map(fold_vaca)).sum())
if distinto_ev:
    problemas.append(f"{distinto_ev} evaluaciones con un pliegue distinto al de su vaca")
huellas = usar.groupby("sha256")["fold"].nunique()
if (huellas > 1).any():
    problemas.append(f"{int((huellas > 1).sum())} fotos idénticas en más de un pliegue")

todas = set(fold_vaca.index)
veces_en_validacion = pd.Series(0, index=sorted(todas))
for k in range(K):
    sin_usar, val, ent = conjuntos(k)
    a = set().union(*(vacas_pliegue[j] for j in ent))
    b, c = vacas_pliegue[val], vacas_pliegue[sin_usar]
    if a & b or a & c or b & c:
        problemas.append(f"rotación {k}: hay vacas en dos conjuntos")
    if a | b | c != todas:
        problemas.append(f"rotación {k}: los conjuntos no cubren todas las vacas")
    veces_en_validacion[list(b)] += 1
if (veces_en_validacion != 1).any():
    problemas.append(f"{int((veces_en_validacion != 1).sum())} vacas no quedan en validación exactamente una vez")

print(f"Vacas revisadas: {len(todas)} | fotos: {len(usar)} | evaluaciones: {len(ev_e)} | rotaciones: {K}")
if problemas:
    raise AssertionError("Hay fuga de datos: " + "; ".join(problemas))
print("Sin fuga de datos. Ninguna vaca ni foto aparece en dos conjuntos y cada vaca se valida una sola vez.")

Vacas revisadas: 76 | fotos: 551 | evaluaciones: 106 | rotaciones: 5
Sin fuga de datos. Ninguna vaca ni foto aparece en dos conjuntos y cada vaca se valida una sola vez.


## Modelo base en cada rotación

En cada rotación se calcula la mediana del BCS de las vacas de entrenamiento (cada vaca cuenta una vez, sin importar cuántas fotos tenga) y se asigna a todas las vacas de validación. Se mide de dos formas.

- Por vaca, contra su BCS de entrenamiento (el promedio de sus evaluaciones). Es la métrica de la fase de modelado y se reporta como vacas dentro de ±0.25 sobre el total.
- Por evaluación, contra cada evaluación real de `evaluaciones.csv`, como se definió en la preparación de los datos.

En las dos se calculan el MAE y el porcentaje dentro de ±0.25 y de ±0.50.

In [6]:
def metricas(err, prefijo):
    m = {f"{prefijo}_n": int(len(err)), f"{prefijo}_mae": round(float(err.mean()), 4)}
    for t in TOLERANCIAS:
        dentro = int((err <= t + 1e-9).sum())
        m[f"{prefijo}_dentro_{t:.2f}"] = dentro
        m[f"{prefijo}_pct_{t:.2f}"] = round(100 * dentro / len(err), 1)
    return m


filas, err_vacas, err_evals = [], [], []
for k in range(K):
    sin_usar, val, ent = conjuntos(k)
    pred = float(vac_e.loc[vac_e["fold"].isin(ent), "bcs_entrenamiento"].median())
    e_vaca = (pred - vac_e.loc[vac_e["fold"] == val, "bcs_entrenamiento"]).abs()
    e_eval = (pred - ev_e.loc[ev_e["fold"] == val, "bcs"]).abs()
    err_vacas.append(e_vaca)
    err_evals.append(e_eval)
    filas.append({"rotacion": k, "prediccion_modelo_base": pred, **metricas(e_vaca, "vaca"), **metricas(e_eval, "eval")})

base = pd.DataFrame(filas).set_index("rotacion")
total = {"prediccion_modelo_base": np.nan, **metricas(pd.concat(err_vacas), "vaca"), **metricas(pd.concat(err_evals), "eval")}
base.loc["total"] = total
display(base)

t = base.loc["total"]
print(f"Modelo base por vaca: {int(t['vaca_dentro_0.25'])}/{int(t['vaca_n'])} dentro de ±0.25 ({t['vaca_pct_0.25']}%), "
      f"MAE {t['vaca_mae']:.3f}, {t['vaca_pct_0.50']}% dentro de ±0.50")
print(f"Modelo base por evaluación: {int(t['eval_dentro_0.25'])}/{int(t['eval_n'])} dentro de ±0.25 "
      f"({t['eval_pct_0.25']}%), MAE {t['eval_mae']:.3f}, {t['eval_pct_0.50']}% dentro de ±0.50")

,prediccion_modelo_base,vaca_n,vaca_mae,vaca_dentro_0.25,vaca_pct_0.25,vaca_dentro_0.50,vaca_pct_0.50,eval_n,eval_mae,eval_dentro_0.25,eval_pct_0.25,eval_dentro_0.50,eval_pct_0.50
rotacion,,,,,,,,,,,,,
0,3.0,15,0.3417,8,53.3,13,86.7,21,0.3571,12,57.1,19,90.5
1,3.0,15,0.3333,8,53.3,13,86.7,19,0.2895,12,63.2,17,89.5
2,3.0,15,0.3167,8,53.3,14,93.3,22,0.2955,13,59.1,20,90.9
3,3.0,15,0.3167,8,53.3,14,93.3,19,0.2763,12,63.2,18,94.7
4,3.0,16,0.3828,8,50.0,13,81.2,25,0.3700,13,52.0,21,84.0
total,NaN,76,0.3388,40,52.6,67,88.2,106,0.3208,62,58.5,95,89.6


Modelo base por vaca: 40/76 dentro de ±0.25 (52.6%), MAE 0.339, 88.2% dentro de ±0.50
Modelo base por evaluación: 62/106 dentro de ±0.25 (58.5%), MAE 0.321, 89.6% dentro de ±0.50


## Archivos de resultados

Se guardan y se descargan dos archivos sin IDs de vacas.

- `resumen_pliegues.csv` tiene una fila por pliegue con vacas, fotos por vista, BCS medio, mínimo y máximo, y evaluaciones.
- `rotaciones_modelo_base.csv` tiene una fila por rotación con sus conjuntos y las métricas del modelo base, más la fila del total.

Los dos van a la carpeta `reports/bigdata/resultados/` del repositorio.

In [7]:
from google.colab import files

carpeta = Path("/content/resultados")
carpeta.mkdir(exist_ok=True)
pliegues.round(4).to_csv(carpeta / "resumen_pliegues.csv")
salida_rot = rotaciones.join(base, how="outer")
for c in rotaciones.columns.drop("pliegues de entrenamiento"):
    salida_rot[c] = salida_rot[c].astype("Int64")
salida_rot.to_csv(carpeta / "rotaciones_modelo_base.csv")
for nombre in ("resumen_pliegues.csv", "rotaciones_modelo_base.csv"):
    files.download(str(carpeta / nombre))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>